# Chapter 13 — Reliability & Failure Probability
*Track 4: Engineers*

## 🎯 Learning Objectives
- Model component failure using probability distributions
- Compute system reliability for series and parallel configurations
- Understand MTTF, MTBF, and failure rate functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Reliability Function

The **reliability function** $R(t)$ is the probability that a component
survives past time $t$:
$$R(t) = P(T > t) = 1 - F(t)$$

The **failure rate** (hazard rate):
$$h(t) = \frac{f(t)}{R(t)}$$

**Bathtub curve** — three phases:
1. **Infant mortality** (decreasing h): early defects
2. **Useful life** (constant h): random failures → Exponential distribution
3. **Wear-out** (increasing h): aging → Weibull distribution

In [ ]:
t = np.linspace(0, 100, 500)

# Bathtub curve (sum of decreasing + constant + increasing hazard)
h_infant   = 0.05 * np.exp(-0.08 * t)
h_useful   = 0.005 * np.ones_like(t)
h_wearout  = 0.002 * np.exp(0.03 * (t - 60)) * (t > 60)
h_bathtub  = h_infant + h_useful + h_wearout

plt.figure(figsize=(10, 4))
plt.plot(t, h_bathtub, lw=2)
plt.fill_between(t[:100], h_bathtub[:100], alpha=0.3, color="red",   label="Infant mortality")
plt.fill_between(t[100:300], h_bathtub[100:300], alpha=0.3, color="green", label="Useful life")
plt.fill_between(t[300:], h_bathtub[300:], alpha=0.3, color="orange", label="Wear-out")
plt.xlabel("Time"); plt.ylabel("Hazard rate h(t)")
plt.title("Bathtub Curve — System Failure Rate over Time")
plt.legend(); plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Exponential & Weibull

**Exponential** (constant hazard, memoryless):
$$R(t) = e^{-\lambda t}, \quad \text{MTTF} = \frac{1}{\lambda}$$

**Weibull** (generalised — shape $k$, scale $\lambda$):
$$R(t) = e^{-(t/\lambda)^k}$$
- $k < 1$: decreasing hazard (infant mortality)
- $k = 1$: constant hazard (exponential)
- $k > 1$: increasing hazard (wear-out)

In [ ]:
t_range = np.linspace(0, 5, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Weibull reliability
for k in [0.5, 1.0, 2.0, 3.5]:
    R = np.exp(-(t_range)**k)
    axes[0].plot(t_range, R, label=f"k={k}")
axes[0].set_xlabel("t/λ"); axes[0].set_ylabel("R(t)")
axes[0].set_title("Weibull Reliability Function"); axes[0].legend()

# Weibull hazard
for k in [0.5, 1.0, 2.0, 3.5]:
    dist = stats.weibull_min(k)
    h = dist.pdf(t_range) / (dist.sf(t_range) + 1e-10)
    axes[1].plot(t_range, h, label=f"k={k}")
axes[1].set_ylim(0, 5); axes[1].set_xlabel("t/λ")
axes[1].set_ylabel("h(t)"); axes[1].set_title("Weibull Hazard Function")
axes[1].legend()
plt.tight_layout(); plt.show()

## 3. Simulation — Series vs Parallel System Reliability

In [ ]:
# Series: system fails if ANY component fails
# Parallel: system fails only if ALL components fail

def series_reliability(component_reliabilities):
    return np.prod(component_reliabilities)

def parallel_reliability(component_reliabilities):
    return 1 - np.prod(1 - np.array(component_reliabilities))

R_comp = 0.95  # individual component reliability
n_range = range(1, 11)

series  = [series_reliability([R_comp]*n) for n in n_range]
parallel = [parallel_reliability([R_comp]*n) for n in n_range]

plt.figure(figsize=(9, 5))
plt.plot(list(n_range), series,   "r-o", label="Series (fails if any fails)")
plt.plot(list(n_range), parallel, "b-o", label="Parallel (fails only if all fail)")
plt.axhline(R_comp, ls="--", color="gray", label=f"Single component R={R_comp}")
plt.xlabel("Number of components"); plt.ylabel("System Reliability")
plt.title("Series vs Parallel System Reliability (R=0.95 per component)")
plt.legend(); plt.tight_layout(); plt.show()

## 4–6. Monte Carlo Simulation, Real Analysis & Practice

In [ ]:
# Monte Carlo life testing simulation
n_components = 1000
mttf = 1000  # hours
failure_times = rng.exponential(mttf, n_components)

print(f"Simulated MTTF: {failure_times.mean():.1f} hours (true={mttf})")
print(f"R(500): simulated={np.mean(failure_times > 500):.4f}, analytic={np.exp(-500/mttf):.4f}")
print(f"R(1000): simulated={np.mean(failure_times > 1000):.4f}, analytic={np.exp(-1):.4f}")

# Fit Weibull to failure data (with wear-out)
failure_wearout = rng.weibull(2.5, n_components) * 800 + 200
shape, loc, scale = stats.weibull_min.fit(failure_wearout, floc=0)
print(f"
Weibull fit: shape={shape:.2f}, scale={scale:.0f}")
print(f"MTTF estimate: {scale * stats.gamma(1 + 1/shape):.0f} hours")

In [ ]:
# Practice P1: Compute MTTF and B10 life for a pump
lam_pump = 1/2000  # failure rate: 1 failure per 2000 hours
MTTF = 1/lam_pump
B10_life = -np.log(0.90) / lam_pump  # time when 10% have failed
print(f"Pump MTTF: {MTTF:.0f} hours")
print(f"B10 life (10% failure): {B10_life:.0f} hours")

# Practice P2: 4-component series system
R_each = np.array([0.99, 0.97, 0.98, 0.995])
R_series = series_reliability(R_each)
print(f"
4-component series reliability: {R_series:.4f}")
print(f"System unreliability: {1-R_series:.4f}")